Vanilla RAG Framework for course notes.
author: Pearl Park

In [1]:
from dotenv import load_dotenv

load_dotenv()  


True

In [2]:
# import chroma for making embeddings:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader = PyPDFLoader("data/cloud_computing.pdf")
pages = loader.load()
print(pages[0].metadata)

/var/folders/kx/g15cvyfn56z4p94t1fz7sjpc0000gn/T/ipykernel_12091/2891031572.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Ignoring wrong pointing object 31 0 (offset 0)


{'producer': 'macOS Version 26.5 (Build 25F71) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20261018225820Z00'00'", 'moddate': "D:20261018225820Z00'00'", 'title': 'CPSC 436C', 'source': 'data/cloud_computing.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


In [3]:
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap= 100,
) 

chunks = pdf_splitter.split_documents(pages)
print(len(chunks))
print(chunks[10].page_content[:200])
print(chunks[10].metadata)


19
○ Storage classes depending on frequency of interaction 
○ Terminate process and save data  
● pay for what you use; GB-month stored, per request, and GB transferred out 
Time and allocation of resour
{'producer': 'macOS Version 26.5 (Build 25F71) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20261018225820Z00'00'", 'moddate': "D:20261018225820Z00'00'", 'title': 'CPSC 436C', 'source': 'data/cloud_computing.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}


In [4]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_vertexai import VertexAIEmbeddings, ChatVertexAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os

PROJECT = os.environ["GCP_PROJECT"]

# embeddings + vector store 
embeddings = VertexAIEmbeddings(model_name="text-embedding-005", project=PROJECT)
vector_store = Chroma(collection_name="lectures", embedding_function=embeddings,
                      persist_directory="vectorstore")

BATCH_SIZE=20
for i in range(0, len(chunks), BATCH_SIZE):
    vector_store.add_documents(chunks[i:i+BATCH_SIZE])


# retriever + LLM
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
llm = ChatVertexAI(model_name="gemini-2.5-flash", project=PROJECT, location="us-central1")

# format docs with citations
def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata['source'].split('/')[-1]}, page {d.metadata['page'] + 1}]\n{d.page_content}"
        for d in docs
    )

/Users/pearl/RAG/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.18). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/pearl/RAG/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.18). Google will not post any further updates to google.cloud.aiplatform_v1beta1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.aiplatform_v1beta1.
  warnings.warn(message, FutureWarning)
/Users/pearl/RAG/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:223: FutureWarning: You are using a non-supported Python vers

In [5]:
prompt = PromptTemplate.from_template("""Use only the context to answer the question.
Cite the file and page for each point. If the answer isn't in the context, say you don't know.

context: {context}

question: {query}

answer: """)

# create the rag_chain
rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is a wide transformation in Spark?"))
print(rag_chain.invoke("What is the purpose of life?"))   # should say it doesn't know

Learning Spark refers to transformations that need rows from other partitions as wide transformations [l6.pdf, page 11]. Spark handles these by moving data first, an operation called Exchange [l6.pdf, page 11].

Examples of wide transformations include:
*   totals per key (groupBy, GROUP BY) [l6.pdf, page 11]
*   join of two large tables [l6.pdf, page 11]
*   global sort (orderBy) [l6.pdf, page 11]
*   value per key written back on every row (window) [l6.pdf, page 11]
I don't know. The provided context does not contain information about the purpose of life.


In [6]:
data = vector_store.get(include=["metadatas"])
from collections import Counter
print(Counter(m["source"] for m in data["metadatas"]))

Counter({'data/cloud_computing.pdf': 391, 'data/l6.pdf': 294})
